# 10. Elasticsearch Backend for Concept Search

---

This notebook demonstrates how to use **Elasticsearch** as your knowledge layer backend for concept search in Portiere.

**Best for:** Teams with existing Elasticsearch infrastructure, large vocabularies, production deployments needing horizontal scalability.

**What you'll learn:**
- Connect Portiere to an external Elasticsearch cluster
- Index OMOP concepts into Elasticsearch
- Run BM25 text search with vocabulary/domain filtering
- Use Elasticsearch in the full mapping pipeline
- Combine Elasticsearch with FAISS in hybrid mode

**Prerequisites:**
- Elasticsearch 8.x running (e.g., `docker run -p 9200:9200 elasticsearch:8.11.0`)
- `pip install portiere-health[elasticsearch]`

## 1. Setup & Installation

In [1]:
# Install Portiere with Elasticsearch support
# pip install portiere-health[elasticsearch]

In [2]:
import json
from pathlib import Path

# Sample data paths
ASSETS_DIR = Path("example_assets/10_elasticsearch_backend")
PATIENTS_CSV = ASSETS_DIR / "patients.csv"
DIAGNOSES_CSV = ASSETS_DIR / "diagnoses.csv"

# Verify source data files exist
for f in [PATIENTS_CSV, DIAGNOSES_CSV]:
    assert f.exists(), f"Missing: {f}"
print("All sample data files found.")

All sample data files found.


In [3]:
from portiere.knowledge import build_knowledge_layer

ATHENA_DIR = Path("example_assets/vocabulary_download_v5")
VOCAB_DIR  = ATHENA_DIR / "indexes"
BM25S_CORPUS = VOCAB_DIR / "concepts.json"
FAISS_INDEX  = VOCAB_DIR / "concepts.index"
FAISS_META   = VOCAB_DIR / "concepts.meta.json"

# Build indexes from Athena on first run (~10-30 min for FAISS embedding)
if not BM25S_CORPUS.exists():
    paths = build_knowledge_layer(
        athena_path=ATHENA_DIR,
        output_path=VOCAB_DIR,
        backend="hybrid",
        vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    )
    print(f"Built indexes: {paths}")
else:
    print(f"Using existing indexes at {VOCAB_DIR}")

CONCEPTS_JSON = BM25S_CORPUS

Using existing indexes at example_assets/vocabulary_download_v5/indexes


## 2. Index Concepts into Elasticsearch

First, we load the sample concepts and index them into Elasticsearch using the `ElasticsearchBackend`.

In [4]:
from portiere.knowledge.elasticsearch_backend import ElasticsearchBackend

# Connect to local Elasticsearch
ES_URL = "http://localhost:9200"
ES_INDEX = "portiere_concepts_demo"

backend = ElasticsearchBackend(
    url=ES_URL,
    index_name=ES_INDEX,
    verify_certs=False,  # For local dev only
)
print(f"Connected to Elasticsearch at {ES_URL}")

ModuleNotFoundError: No module named 'elasticsearch'

In [5]:
# Load concepts from JSON
with open(CONCEPTS_JSON) as f:
    concepts = json.load(f)

print(f"Loaded {len(concepts)} concepts")
print(f"Vocabularies: {set(c['vocabulary_id'] for c in concepts)}")
print(f"Domains: {set(c['domain_id'] for c in concepts)}")

Loaded 623910 concepts
Vocabularies: {'SNOMED', 'LOINC', 'RxNorm'}
Domains: {'Note', 'Drug', 'Spec Anatomic Site', 'Route', 'Meas Value Operator', 'Observation', 'Meas Value', 'Device', 'Condition', 'Language', 'Relationship', 'Specimen', 'Geography', 'Measurement', 'Procedure'}


In [6]:
# Index concepts into Elasticsearch
backend.index_concepts(concepts)
print(f"Indexed {len(concepts)} concepts into '{ES_INDEX}'")

NameError: name 'backend' is not defined

## 3. Search Concepts via Elasticsearch

Elasticsearch provides **BM25 text search** with optional vocabulary and domain filtering.

In [7]:
# Basic search
results = backend.search("diabetes", limit=5)

print("Search: 'diabetes'")
print("-" * 80)
for r in results:
    print(f"  {r['concept_name']:45s}  {r['vocabulary_id']:10s}  score={r['score']:.2f}")

NameError: name 'backend' is not defined

In [8]:
# Search with vocabulary filter (only SNOMED)
results = backend.search("hypertension", vocabularies=["SNOMED"], limit=5)

print("Search: 'hypertension' (SNOMED only)")
print("-" * 80)
for r in results:
    print(f"  {r['concept_name']:45s}  {r['vocabulary_id']:10s}  score={r['score']:.2f}")

NameError: name 'backend' is not defined

In [9]:
# Search with domain filter (only Drugs)
results = backend.search("aspirin", domain="Drug", limit=5)

print("Search: 'aspirin' (Drug domain only)")
print("-" * 80)
for r in results:
    print(f"  {r['concept_name']:45s}  {r['vocabulary_id']:10s}  score={r['score']:.2f}")

NameError: name 'backend' is not defined

In [10]:
# Batch search — efficient multi-search API
queries = ["diabetes", "hypertension", "headache", "metformin", "glucose"]
batch_results = backend.batch_search(queries, limit=3)

print("Batch search results:")
print("=" * 80)
for query, results in zip(queries, batch_results):
    print(f"\n  Query: '{query}'")
    for r in results:
        print(f"    → {r['concept_name']:40s}  {r['vocabulary_id']:10s}  score={r['score']:.2f}")

NameError: name 'backend' is not defined

In [11]:
# Direct concept lookup by ID
concept = backend.get_concept(201826)
print(f"Concept {concept['concept_id']}: {concept['concept_name']} ({concept['vocabulary_id']})")

NameError: name 'backend' is not defined

## 4. Full Pipeline with Elasticsearch Backend

Now let's use Elasticsearch as the knowledge layer in the full Portiere pipeline.

In [12]:
import portiere
from portiere.config import (
    PortiereConfig,
    KnowledgeLayerConfig,
    EmbeddingConfig,
    RerankerConfig,
)
from portiere.engines import PolarsEngine

# Configure with Elasticsearch backend
config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="elasticsearch",
        elasticsearch_url=ES_URL,
        elasticsearch_index=ES_INDEX,
    ),
    reranker=RerankerConfig(provider="none"),
)

project = portiere.init(
    name="ES Backend Demo",
    engine=PolarsEngine(),
    target_model="omop_cdm_v5.4",
    config=config,
)
print(project)

2026-04-16 00:33:16 [info     ] PolarsEngine initialized      


2026-04-16 00:33:16 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


2026-04-16 00:33:16 [info     ] local_storage.project_created  name='ES Backend Demo' project_id=fde5d9f8-8634-4559-8611-247c7643ddad


Project(name='ES Backend Demo', task='standardize', target_model='omop_cdm_v5.4', mode='local')


In [13]:
# Add data sources
patients_source = project.add_source(str(PATIENTS_CSV))
diagnoses_source = project.add_source(str(DIAGNOSES_CSV))

print(f"Patients: {patients_source['row_count']} rows, {len(patients_source['columns'])} columns")
print(f"Diagnoses: {diagnoses_source['row_count']} rows, {len(diagnoses_source['columns'])} columns")

2026-04-16 00:33:16 [info     ] project.source_added           project='ES Backend Demo' source=patients


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:33:19 [info     ] gx_profiler.profiled           columns=11 rows=15 source=patients


2026-04-16 00:33:19 [info     ] project.profiled               source=patients


2026-04-16 00:33:19 [info     ] project.source_added           project='ES Backend Demo' source=diagnoses


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:33:19 [info     ] gx_profiler.profiled           columns=6 rows=20 source=diagnoses


2026-04-16 00:33:19 [info     ] project.profiled               source=diagnoses


Patients: 15 rows, 11 columns
Diagnoses: 20 rows, 6 columns


In [14]:
# Schema mapping
schema_map = project.map_schema(patients_source)
print(schema_map.summary())

2026-04-16 00:33:19 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=omop_cdm_v5.4


2026-04-16 00:33:20 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:33:20 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:33:20 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:33:22 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:33:22 [info     ] Stage 2 complete               auto=10 review=0 total=11


2026-04-16 00:33:22 [info     ] local_storage.schema_mapping_saved items_count=11 project='ES Backend Demo'


2026-04-16 00:33:22 [info     ] project.schema_mapped          auto=10 project='ES Backend Demo' total=11


{'total': 11, 'auto_accepted': 10, 'needs_review': 0, 'approved': 0, 'unmapped': 1, 'auto_rate': 90.9090909090909}


In [15]:
# Concept mapping using Elasticsearch search
concept_map = project.map_concepts(source=diagnoses_source)
print(concept_map.summary())

# Show mapped concepts
for item in concept_map.items[:5]:
    print(f"  {item.source_code:10s} → {item.target_concept_name or 'UNMAPPED':40s} ({item.confidence:.2f})")

2026-04-16 00:33:22 [info     ] project.code_columns_detected  columns=['diagnosis_code']


2026-04-16 00:33:22 [info     ] Stage 3: Concept mapping (local) columns=['diagnosis_code'] knowledge_backend=elasticsearch


2026-04-16 00:33:22 [info     ] Mapping column: diagnosis_code


2026-04-16 00:33:22 [info     ] Found description column: diagnosis_description code_column=diagnosis_code desc_column=diagnosis_description mappings=8


2026-04-16 00:33:22 [info     ] knowledge.creating_backend     backend=elasticsearch url=http://localhost:9200


/Users/kuangsmacbook/Downloads/CUSP/PORTIERE/portiere/src/portiere/engines/polars_engine.py:131: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()


ImportError: elasticsearch package required for Elasticsearch backend. Install with: pip install elasticsearch

## 5. Hybrid Mode: Elasticsearch + FAISS

Combine BM25 (Elasticsearch) with dense vector search (FAISS) for the best of both worlds using Reciprocal Rank Fusion (RRF).

In [16]:
# Hybrid config: Elasticsearch BM25 + FAISS dense vectors
hybrid_config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="hybrid",
        # BM25 via Elasticsearch
        elasticsearch_url=ES_URL,
        elasticsearch_index=ES_INDEX,
        # Dense vectors via FAISS (shared Athena vocabulary)
        faiss_index_path=FAISS_INDEX,
        faiss_metadata_path=FAISS_META,
        # Fusion settings
        fusion_method="rrf",
        rrf_k=60,
    ),
    embedding=EmbeddingConfig(
        provider="huggingface",
        model="cambridgeltl/SapBERT-from-PubMedBERT-fulltext",
    ),
    reranker=RerankerConfig(provider="none"),
)

print("Hybrid config: Elasticsearch (BM25) + FAISS (dense) with RRF fusion")
print(f"  ES URL: {hybrid_config.knowledge_layer.elasticsearch_url}")
print(f"  FAISS index: {hybrid_config.knowledge_layer.faiss_index_path}")
print(f"  Fusion: {hybrid_config.knowledge_layer.fusion_method} (k={hybrid_config.knowledge_layer.rrf_k})")

Hybrid config: Elasticsearch (BM25) + FAISS (dense) with RRF fusion
  ES URL: http://localhost:9200
  FAISS index: example_assets/vocabulary_download_v5/indexes/concepts.index
  Fusion: rrf (k=60)


In [17]:
# Use the hybrid knowledge layer directly
from portiere import build_knowledge_layer

hybrid_layer = build_knowledge_layer(hybrid_config)

# Search with hybrid (BM25 + semantic)
results = hybrid_layer.search("Type 2 diabetes mellitus", limit=5)

print("Hybrid search: 'Type 2 diabetes mellitus'")
print("-" * 80)
for r in results:
    print(f"  {r['concept_name']:45s}  {r['vocabulary_id']:10s}  score={r['score']:.4f}")

TypeError: build_knowledge_layer() missing 1 required positional argument: 'output_path'

## 6. Elasticsearch Configuration Options

The Elasticsearch backend supports several authentication methods:

In [18]:
# Configuration examples (don't run — reference only)

# Basic auth
# backend = ElasticsearchBackend(
#     url="https://es.mycompany.com:9200",
#     basic_auth=("elastic", "changeme"),
#     verify_certs=True,
# )

# API key auth
# backend = ElasticsearchBackend(
#     url="https://es.mycompany.com:9200",
#     api_key="base64-encoded-api-key",
# )

# Elastic Cloud
# backend = ElasticsearchBackend(
#     url="https://my-deployment.es.us-east-1.aws.cloud.es.io:9243",
#     api_key="cloud-api-key",
#     index_name="portiere_concepts",
# )

print("See code comments above for authentication examples.")

See code comments above for authentication examples.


---

## Summary

| Feature | Elasticsearch Backend |
|---------|----------------------|
| Search type | BM25 (lexical) |
| Scalability | Horizontal (cluster) |
| Batch search | Multi-search API (msearch) |
| Auth | Basic, API key, Cloud |
| Best for | Large vocabs, production, existing ES infra |

**Next steps:**
- Combine with FAISS in hybrid mode for best results
- Add a cross-encoder reranker for precision
- See `04_knowledge_backends.ipynb` for a comparison of all backends